<a href="https://colab.research.google.com/github/MatikMo/FSDTask/blob/main/LSTM/FSDTask_LSTM_BCE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM Modell for all Label


In [166]:
#Dependencies, comment out install comments fpr use with GPU

#!pip install comet_ml > /dev/null 2>&1
#import comet_ml
#COMET_API_KEY = "jbaPpXFZ6mRVGWSatsxpOSi7v"
#assert torch.cuda.is_available(), "Please enable GPU from runtime settings"

# Import PyTorch and other relevant libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Sampler, random_split

#To use the periodic plotter
#!pip install mitdeeplearning --quiet
#import mitdeeplearning as mdl

#Numpy, Matplotlib and random
import numpy as np
import numpy.random as rd
import matplotlib.pyplot as plt
import random

#Nice Visualisation
from tqdm import tqdm

#Python OS module
import os

#Pandas to extract Data
import pandas as pd

#For preprocessing the date
from sklearn import preprocessing as pp
from sklearn.model_selection import train_test_split

#To calculate weigts and show confusion matrix
from collections import Counter
from sklearn.metrics import confusion_matrix

Here we introduce the convertion functions of the label


In [167]:
#To convert the models breaking status codes into binary labels
def convert_label_to_binary(label):
    
    binary_mapping = {
        0: torch.tensor([0, 0, 0, 0]),
        1: torch.tensor([0, 0, 1, 0]),
        2: torch.tensor([0, 0, 0, 1]),
        3: torch.tensor([0, 0, 1, 1]),
        4: torch.tensor([1, 0, 0, 0]),
        5: torch.tensor([0, 1, 0, 0]),
        6: torch.tensor([1, 1, 0, 0]),
        7: torch.tensor([1, 0, 1, 0]),
        8: torch.tensor([0, 1, 0, 1]),
        9: torch.tensor([1, 0, 0, 1]),
        10: torch.tensor([0, 1, 1, 0]),
        11: torch.tensor([1, 1, 1, 1])
    }
    return binary_mapping[label]

def binary_to_original_label(binary_labels):
    label_mapping = {
        (0, 0, 0, 0): 0,
        (0, 0, 1, 0): 1,
        (0, 0, 0, 1): 2,
        (0, 0, 1, 1): 3,
        (1, 0, 0, 0): 4,
        (0, 1, 0, 0): 5,
        (1, 1, 0, 0): 6,
        (1, 0, 1, 0): 7,
        (0, 1, 0, 1): 8,
        (1, 0, 0, 1): 9,
        (0, 1, 1, 0): 10,
        (1, 1, 1, 1): 11
    }
    
    original_labels = [label_mapping.get(tuple(map(int, row)), -1) for row in binary_labels]
    
    return torch.tensor(original_labels)

In [168]:
#Data to train and test
df = pd.read_pickle('/kaggle/input/fsddata-input/train.pickle')

#import Data
sensor_data_raw = torch.tensor(df['sensor_data'],dtype=torch.float32)
label_raw = torch.tensor(df['label'])

#sensor data stays as it is, lables are converted into binary labels as given in previous cell
##create sub label
label = torch.stack([convert_label_to_binary(int(val.item())) for val in label_raw])
sensor_data = sensor_data_raw

#to convert back: 
# original = binary_to_original_label(label)
# print(original)  # tensor([1, 4, 8])

## Data Extension

This section is about hard coded data extension and optional

In [ ]:
#To generate more data, we take an existing measurements, scale them slightly in different ways per measurements ax,ay,az,rx,ry,rz
#or shift them as total in time
def generate_data(measurement):
  #varianz
  max_shift = 5 #timesteps back/forward
  min_scale = 0.95
  max_scale = 1.05
  #get time shift
  m_len = measurement.size(dim=0)
  n_measurements = measurement.size(dim=1)
  shift_range = np.concatenate([np.arange(-max_shift,-1),np.arange(1,max_shift)])
  time_shift = random.choice(shift_range)
  shifted_measurement = torch.zeros_like(measurement)

  #shift data and interpolate
  if time_shift < 0:
    shifted_measurement[0:m_len+time_shift] = measurement[-time_shift:m_len]
    #go trough measurements and interpolate some data
    for i in range(n_measurements):
      x = np.array([m_len+time_shift-3, m_len+time_shift-2, m_len+time_shift-1])
      y = np.array([shifted_measurement[x[0]][i],shifted_measurement[x[1]][i],shifted_measurement[x[2]][i]])
      coeffs = np.polyfit(x,y,deg=2)
      f = np.poly1d(coeffs)
      for j in range(m_len+time_shift,m_len):
        shifted_measurement[j][i] = f(j)

  if time_shift > 0:
    shifted_measurement[time_shift:m_len] = measurement[0:m_len-time_shift]
    #go trough measurements and interpolate some data
    for i in range(n_measurements):
      x = np.array([time_shift, time_shift+1, time_shift+2])
      y = np.array([shifted_measurement[x[0]][i],shifted_measurement[x[1]][i],shifted_measurement[x[2]][i]])
      coeffs = np.polyfit(x,y,deg=2)
      f = np.poly1d(coeffs)
      for j in range(0,time_shift):
        shifted_measurement[j][i] = f(j)

  #scale measurements somehow
  shape = (1,6)
  scalars = (max_scale - min_scale) * torch.rand(shape) + min_scale
  shifted_and_scaled = shifted_measurement * scalars

  #test
  #for i,j in [[10,0],[14,1],[20,2],[30,3],[40,4],[50,5]]:
    #print(shifted_and_scaled[i+time_shift][j].item()/measurement[i][j].item())

  return shifted_and_scaled

In [ ]:
#we increase the amount of data by a factor
factor = 4
sensor_data_aug = [sensor_data_raw]
label_aug = [label]

for id, el in enumerate(sensor_data_raw):
    new_samples = []
    new_labels = []
    for _ in range(factor):
        new_data = generate_data(el)
        new_samples.append(new_data.unsqueeze(0))  # (1, 128, 6)
        new_labels.append(label[id].unsqueeze(0))  # (1,)
    sensor_data_aug.append(torch.cat(new_samples, dim=0))
    label_aug.append(torch.cat(new_labels, dim=0))

# Concanate all Parts
sensor_data_full = torch.cat(sensor_data_aug, dim=0)
label_full = torch.cat(label_aug, dim=0)

print(sensor_data_full.shape)
#print(np.bincount(label))

#Save new Dataset
torch.save({
    'features': sensor_data_full,
    'labels': label_full
}, '/kaggle/working/augmented_dataset.pt')


In [ ]:
#Load augmented data
data = torch.load('/kaggle/working/augmented_dataset.pt')
sensor_data = data['features'].to(device)
label = data['labels'].to(device)


## Data Loader and Batching


In [169]:
#split data into validation data and training data and generate data loaders
def split_data(sensor_data, label, rand_seed):
  #use build in test_split method
  sensor_data_train, sensor_data_val, label_train, label_val = train_test_split(
    sensor_data,
    label,
    test_size = 0.25,
    shuffle=True,
    random_state=rand_seed
  )
  #concernate data again to build data loader
  train_data = TensorDataset(sensor_data_train, label_train)
  val_data = TensorDataset(sensor_data_val, label_val)
  return train_data, val_data

def get_loader(train_data, val_data, batch_size):
  #dataloader
  train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=True)

  return train_loader, val_loader


## LSTM Class


In [ ]:
### Defining the LSTM Model ###
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size, batch_size, num_layers, dropout):
        super(LSTMModel, self).__init__()
        # Dimension of hidden layer
        self.hidden_size = hidden_size
        #generate LSTM Network
        self.lstm = nn.LSTM(input_dim, hidden_size, num_layers=num_layers, dropout=dropout, batch_first=True)
        #Linear Dense Layer
        self.fc = nn.Linear(hidden_size,4) #changed for another label-handling

    #Same as in coding lab to initialize LSTM
    def init_hidden(self, batch_size, device):
        # Initialize hidden state and cell state with zeros
        return (torch.zeros(num_layers, batch_size, self.hidden_size).to(device),
                torch.zeros(num_layers, batch_size, self.hidden_size).to(device))

    def forward(self, x, state=None, return_state=False):

        if state is None:
            state = self.init_hidden(x.size(0), x.device)
        out, state = self.lstm(x, state)
        #nur letztes label zählt
        out = self.fc(out[:,-1,:])
        return out if not return_state else (out, state)


Now its already time to think about the parameters, the optimizer and the loss function to train out network.

In [171]:
#Intialize modell
##Measurements have 6 datapoints per timestep
input_dim = sensor_data.size(dim=2)
##Length of one Measurement
seq_length = sensor_data.size(dim=1)
##Number of hidden neurons
hidden_size = 1024
##Number of measuement per batch
batch_size = 64
##Training Iterations and
num_epochs = 5*40
#Checkpoint Infrastructure
checkpoint_dir = '/kaggle/working/training_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
training_attempt = '1024'

#Amount of layers
num_layers = 1
#Dropout
dropout = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMModel(input_dim, hidden_size, batch_size, num_layers, dropout).to(device)
#model.load_state_dict(torch.load('/kaggle/working/training_checkpoints/modellmultAOepoch58acc0.53', weights_only=False))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-5)

#There are around 40% of broken breaks and 60% of working breaks in the data
#We try to tackle this problem in a first approach by giving weights to the loss function

#Calculate weights per label:

#nunm different Label
label_list = [tuple(row.tolist()) for row in label]
counts = Counter(label_list)
raw_weights = {lab: 1.0 / count for lab, count in counts.items()}
mean_val = np.mean(list(raw_weights.values()))
weights = {lab: val / mean_val for lab, val in raw_weights.items()}

def loss_fn(pred, y):
    
    sample_weights = torch.tensor([weights[tuple(label.tolist())] for label in y])

    sample_weights = sample_weights.to(pred.device)
    
    loss_fn_raw = nn.BCEWithLogitsLoss(reduction='none')
    raw_loss = loss_fn_raw(pred, y)  # shape: [batch_size, 4]

    # Mittelwert je Sample berechnen (über Output-Dimensionen)
    loss_per_sample = raw_loss.mean(dim=1)  # shape: [batch_size]
    
    # Gewichtet summieren
    final_loss = (loss_per_sample * sample_weights).mean()

    return final_loss

Training and evaluation functions

In [172]:
def train_step(x, y):
  # Set the model's mode to train
  model.train()
  # Zero gradients for every step
  optimizer.zero_grad()
  # forward and compute loss
  y_hat = model(x)
  loss = loss_fn(y_hat.squeeze(),y.float())
  #gradient computation and optimization
  loss.backward()
  #gradient clipping against exploding gadients
  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
  #step go right, step to left one to the front one to the back, turn the hipps, clap the hands 7 x 7 is finer sand (s)
  optimizer.step()

  return loss

def evaluate(val_loader, set_max, test_eval=False):
    max_batches = round(100 / batch_size) + 1
    model.eval()
    correct, total = 0, 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xv, yv in val_loader:
            max_batches -= 1
            if max_batches < 0 and set_max:
                break

            xv = xv.to(device)
            yv = yv.to(device)

            y_hat = model(xv)
            probs = torch.sigmoid(y_hat)
            pred = (probs > 0.5).long()
            yv = yv.long()

            # Zähle vollständig korrekte Samples
            correct += ((pred == yv).all(dim=1)).sum().item()
            total += yv.size(0)

            # Für Confusion Matrix sammeln
            all_preds.append(pred.cpu())
            all_labels.append(yv.cpu())

    # Alle Samples zu einem Tensor stapeln
    all_preds = torch.cat(all_preds, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy()

    # Kombinierte Label-Klassen (z.B. [1,0,0,1] als Klasse behandeln)
    label_tuples = [tuple(row) for row in all_labels]
    pred_tuples = [tuple(row) for row in all_preds]
    
    # Mapping: Label-Vektor → Integer-Klasse
    all_classes = sorted(set(label_tuples + pred_tuples))
    tuple_to_class = {tpl: idx for idx, tpl in enumerate(all_classes)}
    
    y_true_ids = [tuple_to_class[tpl] for tpl in label_tuples]
    y_pred_ids = [tuple_to_class[tpl] for tpl in pred_tuples]
    
    # Confusion Matrix über kombinierte Klassen
    cm_full = confusion_matrix(y_true_ids, y_pred_ids)

    acc = correct / total

    if test_eval:
        print("Confusion Matrix (kombinierte Klassen):")
        print(cm_full)
        print("\nLabel-Klassen Mapping:")
        for tpl, idx in tuple_to_class.items():
            print(f"Klasse {idx}: {tpl}")

    #To only differ between working/non-working breaks
    zero_label = torch.tensor([0, 0, 0, 0])

    labels_zero = (all_labels == zero_label).all(dim=1)     
    preds_zero = (all_preds == zero_label).all(dim=1)      
    
    labels_nonzero = ~labels_zero                            
    preds_nonzero = ~preds_zero                             
    
    correct_zero = (labels_zero & preds_zero).sum().item()
    
    correct_nonzero = (labels_nonzero & preds_nonzero).sum().item()
    
    total_zero = labels_zero.sum().item()
    total_nonzero = labels_nonzero.sum().item()

    if test_eval: 
        # Ratios
        ratio = (correct_zero + correct_nonzero) / (total_zero + total_nonzero)
        ratio_zero = correct_zero / total_zero if total_zero > 0 else 0
        ratio_nonzero = correct_nonzero / total_nonzero if total_nonzero > 0 else 0

        print(f'Accuracy on working/non-working breaks: {ratio} (accO:{ratio_zero}, acc1:{ratio_nonzero})')

    return acc

## Training Loop


In [173]:
#Move model to GPU and train
model.to(device)

history = []
#plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss')

#initial loader
train_data, val_data = split_data(sensor_data, label, 161)
train_loader, val_loader = get_loader(train_data, val_data, batch_size)

if hasattr(tqdm, '_instances'): tqdm._instances.clear() # clear if it exists
for epoch in range(num_epochs):

    print(f"In Epoch {epoch}/{num_epochs}")

    #count iteration per epoch
    iter = 0
    acc = 0

    #split into new train and val data after 3 epochs
    if epoch % 5 == 0:
      seed = pow(epoch, 7) % 419
      train_data, val_data = split_data(sensor_data, label, seed)
      train_loader, val_loader = get_loader(train_data, val_data, batch_size)

    progress_bar = tqdm(train_loader, desc=f"Progress in Epoch {epoch}", unit="Batch")
    for x,y in progress_bar:

      # Convert numpy arrays to PyTorch tensors
      x_batch = x.clone().detach().to(torch.float32).to(device)
      y_batch = y.clone().detach().to(torch.float32).to(device)

      # Take a train step
      loss = train_step(x_batch, y_batch)

      #to visualize within the plotter
      history.append(loss.item())
      #plotter.plot(history)

      # Print progress
      if iter % 20 == 0:
          acc = evaluate(val_loader, set_max=True, test_eval=False)
      #update progress bar
      progress_bar.set_postfix(accuracy=f"{acc:.2f}")

      #increase iteration index
      iter = iter + 1

    #save modell after each epoch
    checkpoint_prefix = os.path.join(checkpoint_dir, "modell" + str(training_attempt) + "epoch" + str(epoch) + "acc" + str(round(acc,2)))
    torch.save(model.state_dict(), checkpoint_prefix)

    #shuffle train/val data new after each epoch
    train_loader, val_loader = get_loader(train_data, val_data, batch_size)

# Save the final trained model
torch.save(model.state_dict(), checkpoint_prefix)

In Epoch 0/200


Progress in Epoch 0: 100%|██████████| 48/48 [00:02<00:00, 18.94Batch/s, accuracy=0.03]


In Epoch 1/200


Progress in Epoch 1:  38%|███▊      | 18/48 [00:01<00:01, 17.88Batch/s, accuracy=0.01]


KeyboardInterrupt: 

## Here we test the Data


In [ ]:
#Data to train
df.test = pd.read_pickle('/kaggle/input/fsddata-input/test.pickle')

#import Data
sensor_data_test_raw = torch.tensor(df.test['sensor_data'],dtype=torch.float32)
label_test_raw = torch.tensor(df.test['label'])

#conversion and normalisation as above
label_test = torch.stack([convert_label_to_binary(int(val.item())) for val in label_test_raw])
sensor_data_test = sensor_data_test_raw

#concernate and create data loader
test_data = TensorDataset(sensor_data_test, label_test)

Here we can load one ore multiple modells and look into the predictions

In [ ]:
#Loading different training modells
# 
# constant parameters:
#  - Num Layers  ---------- 2
#  - Layer Size ----------- 512
#  - Epochs --------------- 20
#  - Iteration per Epoch -- 3
#
# Model           
# -------------------------------------------------------------------------------------------------------------
# Batch Size
# Learning Rate
# 

hidden_size = 1024
batch_size = 32
num_layers = 1
dropout = 0

#Fraction of Validation Data: 0.25


test_loader = DataLoader(test_data, batch_size=batch_size)

folder_path = '/kaggle/working/training_checkpoints'

for filename in os.listdir(folder_path):
    if filename.startswith('modell1024'):
      file_path = os.path.join(folder_path, filename)
      model = LSTMModel(input_dim, hidden_size, batch_size, num_layers, dropout).to(device)
      model.load_state_dict(torch.load(file_path, weights_only=False))
      acc = evaluate(test_loader, set_max=False, test_eval=False)
      if acc > 0.5:
          print('Accuracy for ' + filename + ' :' + str(acc))
          #evaluate actual effeciency on test data
          print(evaluate(test_loader, set_max=False, test_eval=True))

Helper function to use in caggle

In [ ]:
import shutil
import os

folder_path = '/kaggle/working/training_checkpoints'

# Prüfen, ob der Ordner existiert und dann löschen
if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
    print(f"Ordner '{folder_path}' wurde gelöscht.")
else:
    print(f"Ordner '{folder_path}' existiert nicht.")